<a href="https://colab.research.google.com/github/psvprasad2003/IIMV_CAPSTONE_FILES/blob/IIMV_CAPSTONE_CODE_FILES/04_FEATURE_ENGINEERING_AND_MODEL_DEVELOPMENT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11002BC Hybrid V4B.2: 8004 Operating Model + All-Fault Landing Memory, ver04

**Purpose:** combine the compact V4B operating-model family with corrected V3.2 all-code landing memory.

Matched variants:
- **V3:** calendar-time fault history
- **V3.2-500 / V3.2-1000:** V3 plus detailed and all-code landing memory
- **V4B:** V3 plus compact 8004 operating-model features
- **V4B.2-500 / V4B.2-1000:** V3.2 plus V4B

Parent branches are isolated explicitly. All rows, labels, groups, folds, and learner hyperparameters remain matched. Current-landing fault bins are excluded from landing-memory predictors. Cross-side and enriched matrices remain in memory and are not persisted.


## Configuration

Resolves the most recent Part B run under `11002BC`'s output tree (same
resume convention as the notebook's own CELL 01 fix -- never mints a new
run directory, since this builds on an already-completed Part B pass).

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ProcessPoolExecutor, as_completed
import json
import os
import re
import shutil
import time
import traceback

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

WORK_ROOT = Path("/raid3/e296408/All_ECFRs_working")
ROW_V2_ROOT = WORK_ROOT / "ECFR_11001_ROW_V2" / "records"
BC_ROOT = WORK_ROOT / "ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED"
ENGINE_ROOT = BC_ROOT / "engines"

# Resume convention: reuse the most recently modified existing Part B run,
# same as the notebook's own CELL 01 fix -- never mints a new run directory,
# since this script is meant to build on an already-completed Part B pass.
_data_richness_root = BC_ROOT / "data_richness"
_existing_runs = (
    sorted(_data_richness_root.glob("run_*"), key=lambda p: p.stat().st_mtime)
    if _data_richness_root.is_dir() else []
)
if not _existing_runs:
    raise RuntimeError(
        f"No existing Part B run found under {_data_richness_root} -- "
        f"run 11002BC through CELL 04 (feature build) first."
    )
PART_B_RUN_ROOT = _existing_runs[-1]
DATA_ROOT = PART_B_RUN_ROOT / "data"
MODEL_INPUT_FILE = DATA_ROOT / "landing_level_model_inputs.parquet"
if not MODEL_INPUT_FILE.is_file():
    raise FileNotFoundError(MODEL_INPUT_FILE)

V1_OUTPUT_ROOT = PART_B_RUN_ROOT / "hybrid_trailing_history_benchmark"
OUTPUT_ROOT = PART_B_RUN_ROOT / "hybrid_v4b_2_8004_all_fault_memory_ver04"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Reuse the already-computed V1 trailing cache; do not recompute completed work.
TRAILING_CACHE_ROOT = V1_OUTPUT_ROOT / "trailing_features_by_engine"
FAULT_CACHE_ROOT = PART_B_RUN_ROOT / "hybrid_v3_fault_history_benchmark" / "fault_history_by_engine"
FAULT_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
CANONICAL_FAULT_FILE = DATA_ROOT / "canonical_fault_events.parquet"
LANDING_MEMORY_CONTRACT = "HYBRID_V3_2_ALL_FAULT_LANDING_MEMORY_VER05"
LANDING_MEMORY_CACHE_ROOT = PART_B_RUN_ROOT / "_shared_feature_cache" / LANDING_MEMORY_CONTRACT
LANDING_MEMORY_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
TRAILING_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

LOOKBACK_DAYS = 30          # matches 11007a
HORIZON_DAYS = 14           # matches both notebooks
FEATURE_CUTOFF_SECONDS = 900  # matches 11002BC's existing landing-window cutoff
TOP_N_FEATURES = 500
TOP_N_FEATURES_EXPANDED = 1000        # same cap as before, but now RANKED, not arbitrary
ENGINE_WORKERS = min(50, os.cpu_count() or 8)
MIN_TS = pd.Timestamp("2000-01-01", tz="UTC")
MAX_TS = pd.Timestamp(datetime.now(timezone.utc))
STARTUP_TRANSIENT_CODES = {175, 176}
RANDOM_STATE = 11002
LANDING_MEMORY_LAGS = 5
LANDING_MEMORY_WORKERS = min(50, os.cpu_count() or 8)
STATE_CACHE_CONTRACT = "V4B_COMPACT_8004_OPMODEL_V3_DWELLFIX"
STATE_CACHE_ROOT = PART_B_RUN_ROOT / "_shared_feature_cache" / STATE_CACHE_CONTRACT
STATE_CACHE_MANIFEST = STATE_CACHE_ROOT / "cache_manifest.parquet"
STATE_WORKERS = max(1, min(8, ENGINE_WORKERS))
MIN_FREE_GIB = 50.0
MAX_STATE_CACHE_GIB = 8.0
DEFINITION_PATH = Path("/raid3/e296408/converted_data.xlsx")
NOMINAL_SAMPLE_ECU = 1.0
MAX_CYCLE_ECU = 40.0
STATE_LOOKBACK_DAYS = (7, 30, 90)
STATE_NAMES = {0:"CLOSED",1:"OPENING",2:"OPEN",3:"CLOSING",4:"INHIBITED",5:"FAILED",6:"STATE_6",7:"STATE_7"}
STATE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
LANDING_MEMORY_WEIGHTS = np.array([1.0, 0.80, 0.60, 0.40, 0.20], dtype=np.float32)
RECOVERY_CLEAN_LANDINGS = 3
LOOKBACK_NS = np.int64(pd.Timedelta(days=LOOKBACK_DAYS).value)

print({
    "part_b_run_root": str(PART_B_RUN_ROOT),
    "model_input_file": str(MODEL_INPUT_FILE),
    "v1_cache_root": str(TRAILING_CACHE_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "engine_workers": ENGINE_WORKERS,
    "fault_cache_root": str(FAULT_CACHE_ROOT),
    "landing_memory_cache_root": str(LANDING_MEMORY_CACHE_ROOT),
    "landing_memory_contract": LANDING_MEMORY_CONTRACT,
    "landing_memory_workers": LANDING_MEMORY_WORKERS,
    "top_n_features_expanded": TOP_N_FEATURES_EXPANDED,
})

## Trailing Window Features

The core new logic. For each landing, computes trailing 30-day aggregates
per status field using `searchsorted` + cumulative sums -- not a per-landing
scan, since a naive approach at over a million landings x hundreds of
fields would be far too slow.

**Tested against a brute-force per-window reference** before being trusted
here (realistic random histories, landings before/after all history, an
engine with zero events, an all-NaN field, and a performance check at
20,000 events x 2,000 landings -- 2.2ms).

In [ ]:
# =============================================================================
# TRAILING WINDOW FEATURES (tested against a brute-force reference before
# being used here -- see test_trailing_window.py)
# =============================================================================
def trailing_window_features(landing_times_ns, event_times_ns, event_values, lookback_ns):
    """
    landing_times_ns: (M,) int64 ns, landing timestamps for one engine
    event_times_ns:   (N,) int64 ns, SORTED ascending, this engine's event times
    event_values:     (N,) float, this status field's value per event (NaN where absent)
    lookback_ns:      int64 ns window length

    Returns obs/active/last/changes arrays, one value per landing. Window is
    (landing - lookback, landing], matching 11007a's own window definition
    exactly, including its window-local "changes" semantics (the first
    non-null value observed within a window never counts as a change, even
    if it differs from a value just outside the window) -- confirmed via
    a brute-force per-window reference test, which caught an earlier,
    subtly different global-running version of this same function.
    """
    n = len(event_times_ns)
    if n == 0:
        m = len(landing_times_ns)
        zeros = np.zeros(m, dtype=np.int64)
        return {"obs": zeros, "active": zeros.copy(), "last": np.full(m, np.nan), "changes": zeros.copy()}

    notna = ~np.isnan(event_values)
    active = notna & (event_values != 0)

    cum_obs = np.concatenate([[0], np.cumsum(notna)])
    cum_active = np.concatenate([[0], np.cumsum(active)])

    last_nonnull_idx = np.maximum.accumulate(np.where(notna, np.arange(n), -1))

    next_nonnull_idx_raw = np.where(notna, np.arange(n), n)
    next_nonnull_idx_raw = np.minimum.accumulate(next_nonnull_idx_raw[::-1])[::-1]
    next_nonnull_idx = np.concatenate([next_nonnull_idx_raw, [n]])

    changed = np.zeros(n, dtype=bool)
    prev_nonnull_event_idx = np.full(n, -1, dtype=np.int64)
    seen_vals = event_values[notna]
    seen_idx = np.nonzero(notna)[0]
    if len(seen_vals) > 1:
        diffs = seen_vals[1:] != seen_vals[:-1]
        changed[seen_idx[1:]] = diffs
        prev_nonnull_event_idx[seen_idx[1:]] = seen_idx[:-1]
    cum_changes = np.concatenate([[0], np.cumsum(changed)])

    start_idx = np.searchsorted(event_times_ns, landing_times_ns - lookback_ns, side="right")
    end_idx = np.searchsorted(event_times_ns, landing_times_ns, side="right")

    obs = cum_obs[end_idx] - cum_obs[start_idx]
    act = cum_active[end_idx] - cum_active[start_idx]
    chg_raw = cum_changes[end_idx] - cum_changes[start_idx]

    first_nonnull_in_window = next_nonnull_idx[start_idx]
    has_first = first_nonnull_in_window < end_idx
    safe_first = np.clip(first_nonnull_in_window, 0, n - 1)
    boundary_crossing = (
        has_first
        & changed[safe_first]
        & (prev_nonnull_event_idx[safe_first] < start_idx)
    )
    chg = chg_raw - boundary_crossing.astype(np.int64)

    last_idx_in_window = np.where(end_idx > 0, last_nonnull_idx[np.maximum(end_idx - 1, 0)], -1)
    has_last = (end_idx > start_idx) & (last_idx_in_window >= start_idx)
    last_val = np.where(has_last, event_values[np.clip(last_idx_in_window, 0, n - 1)], np.nan)

    return {"obs": obs, "active": act, "last": last_val, "changes": chg}

## Phase 1: Load model_input, discover status fields

Loads the existing landing-level feature table and discovers every
decoded `sw1_`-`sw4_` status field across a sample of engines -- the same
discovery pattern `11007a` and `11002BC`'s own CELL 03 already use.

In [ ]:
# =============================================================================
# PHASE 1: load model_input, discover status fields to build trailing
# history for (same discovery pattern as 11007a/11002BC's own CELL 03)
# =============================================================================
model_input = pd.read_parquet(MODEL_INPUT_FILE)
model_input["landing_datetime_utc"] = pd.to_datetime(model_input["landing_datetime_utc"], errors="coerce", utc=True)
model_input = model_input.loc[
    model_input["landing_datetime_utc"].notna()
    & model_input["landing_datetime_utc"].ge(MIN_TS)
    & model_input["landing_datetime_utc"].le(MAX_TS)
].reset_index(drop=True)

status_fields = set()
_sample_engine_dirs = list(ENGINE_ROOT.glob("engine=*"))[:50]
for d in _sample_engine_dirs:
    hist_file = d / "history_exact_deduplicated.parquet"
    if hist_file.is_file():
        status_fields.update(
            c for c in pq.ParquetFile(hist_file).schema_arrow.names if re.match(r"^sw[1-4]_", str(c), flags=re.I)
        )
status_fields = sorted(status_fields)
if not status_fields:
    raise RuntimeError("No sw1-4 status fields discovered -- check ENGINE_ROOT / a completed CELL 04 run")

print({"model_input_rows": len(model_input), "engines": model_input["engine_serial"].nunique(),
       "status_fields_discovered": len(status_fields)})

## Phase 2: Per-engine trailing-window computation (parallel)

Each worker loads **one engine's** raw history once and computes all
status fields' trailing features at all of that engine's landings in a
single pass, using the tested vectorized function above. Parallelized
across engines; restartable -- an engine whose cached output already
exists is skipped.

In [ ]:
# =============================================================================
# PHASE 2: per-engine trailing-window feature computation (parallel across
# engines -- each worker loads its OWN engine's history once and computes
# all status fields' trailing features at all that engine's landings in one
# pass, using the tested vectorized function above)
# =============================================================================
def _compute_engine_trailing_features(args):
    engine, landing_ids, landing_times_ns, status_fields_list, engine_root_text, lookback_ns, cache_root_text = args
    started = time.time()
    result = {"engine_serial": engine, "status": "FAIL", "error": "", "rows": 0}
    out_path = Path(cache_root_text) / f"engine={engine}.parquet"

    try:
        hist_path = Path(engine_root_text) / f"engine={engine}" / "history_exact_deduplicated.parquet"
        if not hist_path.is_file():
            raise RuntimeError("No history file for this engine")

        schema_names = pq.ParquetFile(hist_path).schema_arrow.names
        available_fields = [f for f in status_fields_list if f in schema_names]
        if not available_fields or "record_datetime_utc" not in schema_names:
            out = pd.DataFrame({"engine_channel_landing_id": landing_ids})
            out.to_parquet(out_path, index=False, compression="zstd")
            result.update(status="PASS", rows=len(out), elapsed_seconds=round(time.time() - started, 3))
            return result

        hist = pd.read_parquet(hist_path, columns=["record_datetime_utc"] + available_fields)
        event_times = pd.to_datetime(hist["record_datetime_utc"], errors="coerce", utc=True)
        valid = event_times.notna()
        hist = hist.loc[valid].copy()
        event_times = event_times.loc[valid]
        # Converting to int64 while still a pandas Series (which correctly
        # handles the tz-aware -> epoch-ns conversion) before going to numpy
        # -- a plain .to_numpy() on a tz-aware Series can otherwise return an
        # object array of Timestamps depending on pandas version (caught by
        # testing).
        event_times_ns_unsorted = event_times.astype("int64").to_numpy()
        order = np.argsort(event_times_ns_unsorted)
        event_times_ns = event_times_ns_unsorted[order]
        hist = hist.iloc[order].reset_index(drop=True)

        out = {"engine_channel_landing_id": landing_ids}
        for field in available_fields:
            values = pd.to_numeric(hist[field], errors="coerce").to_numpy(dtype=float)
            feats = trailing_window_features(landing_times_ns, event_times_ns, values, lookback_ns)
            out[f"TRAIL30_{field}__OBS"] = feats["obs"]
            out[f"TRAIL30_{field}__ACTIVE"] = feats["active"]
            out[f"TRAIL30_{field}__LAST"] = feats["last"]
            out[f"TRAIL30_{field}__CHANGES"] = feats["changes"]

        out_df = pd.DataFrame(out)
        out_df.to_parquet(out_path, index=False, compression="zstd")
        result.update(status="PASS", rows=len(out_df), elapsed_seconds=round(time.time() - started, 3))

    except Exception:
        result["error"] = traceback.format_exc(limit=8)
        result["elapsed_seconds"] = round(time.time() - started, 3)
    return result


engines_to_process = sorted(model_input["engine_serial"].astype(str).unique())
tasks = []
for engine in engines_to_process:
    marker = TRAILING_CACHE_ROOT / f"engine={engine}.parquet"
    if marker.is_file():
        continue  # restartable: an engine already computed is skipped
    rows = model_input.loc[
        model_input["engine_serial"].astype(str) == engine, ["engine_channel_landing_id", "landing_datetime_utc"]
    ]
    # Passed directly as plain arrays, not round-tripped through JSON --
    # a to_json/read_json round-trip silently mangled the string ID
    # column in testing (pandas' JSON type inference is not reliable for
    # ID-like strings), and ProcessPoolExecutor already pickles numpy
    # arrays and DataFrames correctly across processes, so the JSON step
    # was both unnecessary and the actual source of a real bug.
    landing_ids = rows["engine_channel_landing_id"].astype(str).to_numpy()
    landing_times_ns = pd.to_datetime(rows["landing_datetime_utc"], utc=True).astype("int64").to_numpy()
    tasks.append((engine, landing_ids, landing_times_ns, status_fields, str(ENGINE_ROOT),
                  LOOKBACK_NS, str(TRAILING_CACHE_ROOT)))

print({"engines_total": len(engines_to_process), "already_cached": len(engines_to_process) - len(tasks),
       "remaining": len(tasks)})

trailing_results = []
if tasks:
    with ProcessPoolExecutor(max_workers=ENGINE_WORKERS) as executor:
        futures = {executor.submit(_compute_engine_trailing_features, t): t[0] for t in tasks}
        for i, future in enumerate(as_completed(futures), 1):
            engine = futures[future]
            try:
                r = future.result()
            except Exception:
                r = {"engine_serial": engine, "status": "FAIL", "error": traceback.format_exc(limit=8)}
            trailing_results.append(r)
            if i == 1 or i % 100 == 0 or i == len(tasks):
                print(f"[TRAIL {i:,}/{len(tasks):,}] {engine}: {r['status']}")

if trailing_results:
    failed = [r for r in trailing_results if r["status"] != "PASS"]
    if failed:
        pd.DataFrame(failed).to_csv(OUTPUT_ROOT / "trailing_feature_failures.csv", index=False)
        print(f"WARNING: {len(failed)} engine(s) failed trailing-feature computation -- see trailing_feature_failures.csv")

## Phase 3: Consolidate and join

Concatenates every engine's cached trailing-feature file and joins it
onto `model_input` by `engine_channel_landing_id`. The join key is forced
to a consistent string dtype on both sides -- defensive, since this is
exactly where a real bug surfaced during testing.

In [ ]:
# =============================================================================
# PHASE 3: consolidate trailing features, join onto model_input
# =============================================================================
trailing_files = sorted(TRAILING_CACHE_ROOT.glob("engine=*.parquet"))
trailing_frames = [pd.read_parquet(f) for f in trailing_files]
trailing_all = pd.concat(trailing_frames, ignore_index=True, sort=False) if trailing_frames else pd.DataFrame()

# defensive: force a consistent string dtype for the join key on both
# sides -- the JSON round-trip through the worker processes can otherwise
# leave this column's dtype ambiguous (e.g. inferred as int64 if the IDs
# happen to look numeric), breaking the merge below with a dtype mismatch
model_input["engine_channel_landing_id"] = model_input["engine_channel_landing_id"].astype(str)
if not trailing_all.empty:
    trailing_all["engine_channel_landing_id"] = trailing_all["engine_channel_landing_id"].astype(str)

enriched = model_input.merge(trailing_all, on="engine_channel_landing_id", how="left")
trailing_cols = [c for c in enriched.columns if c.startswith("TRAIL30_")]
print({"trailing_feature_columns_added": len(trailing_cols), "enriched_rows": len(enriched)})

## Phase 3B: Cross-side trailing history and asymmetry

Pairs the focal engine with its resolved cross-side engine at the same aircraft landing event. For every focal trailing-history feature, it adds the partner value, signed focal-minus-partner delta, and absolute delta.


In [ ]:
# =============================================================================
# PHASE 3B: exact-event cross-side trailing history + deltas
# =============================================================================
required_cross_side_columns = {
    "aircraft_landing_event_id", "engine_serial", "cross_side_engine_serial"
}
missing = required_cross_side_columns - set(enriched.columns)
if missing:
    raise RuntimeError(f"Cross-side experiment requires columns: {sorted(missing)}")

# Normalize IDs without converting missing values to the literal string "nan".
for c in ["aircraft_landing_event_id", "engine_serial", "cross_side_engine_serial"]:
    enriched[c] = enriched[c].astype("string").str.strip()

enriched["XSIDE_AVAILABLE"] = (
    enriched["aircraft_landing_event_id"].notna()
    & enriched["cross_side_engine_serial"].notna()
).astype("int8")

# One lookup record per aircraft event and engine. Channel duplicates should have
# identical trailing values because all values are computed as of the same landing.
lookup_columns = ["aircraft_landing_event_id", "engine_serial"] + trailing_cols
cross_lookup = enriched[lookup_columns].drop_duplicates(
    ["aircraft_landing_event_id", "engine_serial"], keep="first"
)
rename_map = {"engine_serial": "cross_side_engine_serial"}
rename_map.update({c: f"XSIDE_{c}" for c in trailing_cols})
cross_lookup = cross_lookup.rename(columns=rename_map)

before_rows = len(enriched)
enriched = enriched.merge(
    cross_lookup,
    on=["aircraft_landing_event_id", "cross_side_engine_serial"],
    how="left",
    validate="many_to_one",
)
if len(enriched) != before_rows:
    raise RuntimeError("Cross-side join changed row count")

xside_cols = [f"XSIDE_{c}" for c in trailing_cols]
# Use float32 blocks to control memory. Missing partner values remain NaN.
focal_block = enriched[trailing_cols].apply(pd.to_numeric, errors="coerce").astype("float32")
xside_block = enriched[xside_cols].apply(pd.to_numeric, errors="coerce").astype("float32")
xside_block.columns = trailing_cols
signed_delta = focal_block - xside_block
absolute_delta = signed_delta.abs()
signed_delta.columns = [f"DELTA_{c}" for c in trailing_cols]
absolute_delta.columns = [f"ABSDELTA_{c}" for c in trailing_cols]
enriched = pd.concat([enriched, signed_delta, absolute_delta], axis=1)

matched_rows = int(enriched[xside_cols].notna().any(axis=1).sum())
matched_events = int(enriched.loc[
    enriched[xside_cols].notna().any(axis=1), "aircraft_landing_event_id"
].nunique())

cross_side_audit = {
    "rows": len(enriched),
    "rows_with_declared_cross_side": int(enriched["XSIDE_AVAILABLE"].sum()),
    "rows_with_matched_cross_side_history": matched_rows,
    "aircraft_events_with_matched_cross_side_history": matched_events,
    "focal_trailing_columns": len(trailing_cols),
    "cross_side_columns": len(xside_cols),
    "signed_delta_columns": len(signed_delta.columns),
    "absolute_delta_columns": len(absolute_delta.columns),
}
(OUTPUT_ROOT / "cross_side_audit.json").write_text(
    json.dumps(cross_side_audit, indent=2), encoding="utf-8"
)
print(cross_side_audit)

del cross_lookup, focal_block, xside_block, signed_delta, absolute_delta


## Phase 3C: Leakage-safe fault progression history

Computes focal-engine fault recency, frequency, diversity, persistence, and velocity at each landing cutoff. It uses each engine's exact deduplicated history, filters to the TR-family code set observed in the canonical fault artifact, and caches one Parquet file per engine. It then adds same-event cross-side fault values and deltas.


In [ ]:
# =============================================================================
# PHASE 3C: focal + cross-side fault progression history
# =============================================================================
if not CANONICAL_FAULT_FILE.is_file():
    raise FileNotFoundError(CANONICAL_FAULT_FILE)
canonical_faults = pd.read_parquet(CANONICAL_FAULT_FILE, columns=["fault_code"])
TR_FAULT_CODES = set(pd.to_numeric(canonical_faults["fault_code"], errors="coerce").dropna().astype(int))
TR_FAULT_CODES -= STARTUP_TRANSIENT_CODES
if not TR_FAULT_CODES:
    raise RuntimeError("No TR-family fault codes were resolved from canonical_fault_events.parquet")

FAULT_LOOKBACK_DAYS = (7, 30, 90)
DAY_NS = np.int64(pd.Timedelta(days=1).value)

def fault_history_features(anchor_ns, event_ns, fault_codes, tr_code_set):
    """Return leakage-safe features using events <= each anchor."""
    m = len(anchor_ns)
    result = {
        "FAULTHIST_DAYS_SINCE_LAST_206": np.full(m, np.nan, dtype=np.float32),
        "FAULTHIST_DAYS_SINCE_LAST_TR": np.full(m, np.nan, dtype=np.float32),
    }
    for days in FAULT_LOOKBACK_DAYS:
        result[f"FAULTHIST_TR_COUNT_{days}D"] = np.zeros(m, dtype=np.int32)
        result[f"FAULTHIST_206_COUNT_{days}D"] = np.zeros(m, dtype=np.int32)
        result[f"FAULTHIST_DISTINCT_TR_{days}D"] = np.zeros(m, dtype=np.int16)
        result[f"FAULTHIST_MAX_REPEAT_TR_{days}D"] = np.zeros(m, dtype=np.int16)
    if len(event_ns) == 0:
        result["FAULTHIST_TR_RATE_7D"] = np.zeros(m, dtype=np.float32)
        result["FAULTHIST_TR_RATE_30D"] = np.zeros(m, dtype=np.float32)
        result["FAULTHIST_TR_ACCEL_7D_VS_30D"] = np.zeros(m, dtype=np.float32)
        result["FAULTHIST_206_ACCEL_7D_VS_30D"] = np.zeros(m, dtype=np.float32)
        return result

    order = np.argsort(event_ns)
    event_ns = event_ns[order]
    fault_codes = fault_codes[order]
    tr_mask = np.isin(fault_codes, list(tr_code_set))
    tr_ns, tr_codes = event_ns[tr_mask], fault_codes[tr_mask]
    f206_ns = event_ns[fault_codes == 206]

    end_tr = np.searchsorted(tr_ns, anchor_ns, side="right")
    end_206 = np.searchsorted(f206_ns, anchor_ns, side="right")
    has_tr = end_tr > 0
    has_206 = end_206 > 0
    result["FAULTHIST_DAYS_SINCE_LAST_TR"][has_tr] = (
        anchor_ns[has_tr] - tr_ns[end_tr[has_tr] - 1]
    ) / DAY_NS
    result["FAULTHIST_DAYS_SINCE_LAST_206"][has_206] = (
        anchor_ns[has_206] - f206_ns[end_206[has_206] - 1]
    ) / DAY_NS

    for days in FAULT_LOOKBACK_DAYS:
        width = np.int64(pd.Timedelta(days=days).value)
        start_tr = np.searchsorted(tr_ns, anchor_ns - width, side="right")
        start_206 = np.searchsorted(f206_ns, anchor_ns - width, side="right")
        result[f"FAULTHIST_TR_COUNT_{days}D"] = (end_tr - start_tr).astype(np.int32)
        result[f"FAULTHIST_206_COUNT_{days}D"] = (end_206 - start_206).astype(np.int32)
        distinct = result[f"FAULTHIST_DISTINCT_TR_{days}D"]
        max_repeat = result[f"FAULTHIST_MAX_REPEAT_TR_{days}D"]
        for i, (left, right) in enumerate(zip(start_tr, end_tr)):
            if right <= left:
                continue
            _, counts = np.unique(tr_codes[left:right], return_counts=True)
            distinct[i] = len(counts)
            max_repeat[i] = int(counts.max())

    result["FAULTHIST_TR_RATE_7D"] = result["FAULTHIST_TR_COUNT_7D"].astype(np.float32) / 7.0
    result["FAULTHIST_TR_RATE_30D"] = result["FAULTHIST_TR_COUNT_30D"].astype(np.float32) / 30.0
    result["FAULTHIST_TR_ACCEL_7D_VS_30D"] = (
        result["FAULTHIST_TR_RATE_7D"] / (result["FAULTHIST_TR_RATE_30D"] + np.float32(1e-3))
    )
    result["FAULTHIST_206_ACCEL_7D_VS_30D"] = (
        (result["FAULTHIST_206_COUNT_7D"].astype(np.float32) / 7.0)
        / ((result["FAULTHIST_206_COUNT_30D"].astype(np.float32) / 30.0) + np.float32(1e-3))
    )
    return result


def compute_engine_fault_history(args):
    engine, landing_ids, cutoff_ns, engine_root_text, cache_root_text, tr_codes = args
    out_path = Path(cache_root_text) / f"engine={engine}.parquet"
    result = {"engine_serial": engine, "status": "FAIL", "rows": 0, "error": ""}
    try:
        hist_path = Path(engine_root_text) / f"engine={engine}" / "history_exact_deduplicated.parquet"
        if not hist_path.is_file():
            raise FileNotFoundError(hist_path)
        names = set(pq.ParquetFile(hist_path).schema_arrow.names)
        required = {"record_datetime_utc", "fault_code"}
        if not required.issubset(names):
            frame = pd.DataFrame({"engine_channel_landing_id": landing_ids})
            for k, v in fault_history_features(cutoff_ns, np.array([], dtype=np.int64), np.array([], dtype=np.int64), tr_codes).items():
                frame[k] = v
        else:
            hist = pd.read_parquet(hist_path, columns=["record_datetime_utc", "fault_code"])
            ts = pd.to_datetime(hist["record_datetime_utc"], errors="coerce", utc=True)
            codes = pd.to_numeric(hist["fault_code"], errors="coerce")
            valid = ts.notna() & codes.notna()
            ev_ns = ts.loc[valid].astype("int64").to_numpy(dtype=np.int64)
            ev_codes = codes.loc[valid].astype(int).to_numpy(dtype=np.int64)
            # Exact event-level de-duplication within this feature family.
            if len(ev_ns):
                pairs = np.rec.fromarrays([ev_ns, ev_codes], names="t,c")
                _, keep = np.unique(pairs, return_index=True)
                ev_ns, ev_codes = ev_ns[keep], ev_codes[keep]
            frame = pd.DataFrame({"engine_channel_landing_id": landing_ids})
            for k, v in fault_history_features(cutoff_ns, ev_ns, ev_codes, tr_codes).items():
                frame[k] = v
        frame.to_parquet(out_path, index=False, compression="zstd")
        result.update(status="PASS", rows=len(frame))
    except Exception:
        result["error"] = traceback.format_exc(limit=8)
    return result

# The feature cutoff is landing + 900 s, matching the FF206 target boundary.
enriched["feature_cutoff_utc"] = enriched["landing_datetime_utc"] + pd.to_timedelta(FEATURE_CUTOFF_SECONDS, unit="s")
fault_tasks = []
for engine, rows in enriched.groupby(enriched["engine_serial"].astype(str), sort=False):
    marker = FAULT_CACHE_ROOT / f"engine={engine}.parquet"
    if marker.is_file():
        continue
    fault_tasks.append((
        str(engine),
        rows["engine_channel_landing_id"].astype(str).to_numpy(),
        rows["feature_cutoff_utc"].astype("int64").to_numpy(dtype=np.int64),
        str(ENGINE_ROOT), str(FAULT_CACHE_ROOT), TR_FAULT_CODES,
    ))
print({"fault_engines_total": enriched["engine_serial"].nunique(), "fault_engines_remaining": len(fault_tasks)})

fault_results = []
if fault_tasks:
    with ProcessPoolExecutor(max_workers=ENGINE_WORKERS) as executor:
        futures = {executor.submit(compute_engine_fault_history, t): t[0] for t in fault_tasks}
        for i, future in enumerate(as_completed(futures), 1):
            engine = futures[future]
            try:
                r = future.result()
            except Exception:
                r = {"engine_serial": engine, "status": "FAIL", "error": traceback.format_exc(limit=8)}
            fault_results.append(r)
            if i == 1 or i % 100 == 0 or i == len(fault_tasks):
                print(f"[FAULT {i:,}/{len(fault_tasks):,}] {engine}: {r['status']}")
if fault_results:
    failed = [r for r in fault_results if r["status"] != "PASS"]
    if failed:
        pd.DataFrame(failed).to_csv(OUTPUT_ROOT / "fault_history_failures.csv", index=False)
        raise RuntimeError(f"{len(failed)} engines failed fault-history generation")

fault_files = sorted(FAULT_CACHE_ROOT.glob("engine=*.parquet"))
fault_all = pd.concat([pd.read_parquet(p) for p in fault_files], ignore_index=True, sort=False)
fault_all["engine_channel_landing_id"] = fault_all["engine_channel_landing_id"].astype(str)
before_rows = len(enriched)
enriched["engine_channel_landing_id"] = enriched["engine_channel_landing_id"].astype(str)
enriched = enriched.merge(fault_all, on="engine_channel_landing_id", how="left", validate="one_to_one")
if len(enriched) != before_rows:
    raise RuntimeError("Fault-history join changed row count")
fault_cols = [c for c in enriched.columns if c.startswith("FAULTHIST_")]

# Cross-side fault history at the same aircraft event.
fault_lookup = enriched[["aircraft_landing_event_id", "engine_serial"] + fault_cols].drop_duplicates(
    ["aircraft_landing_event_id", "engine_serial"], keep="first"
).rename(columns={"engine_serial": "cross_side_engine_serial", **{c: f"XSIDE_{c}" for c in fault_cols}})
enriched = enriched.merge(
    fault_lookup,
    on=["aircraft_landing_event_id", "cross_side_engine_serial"],
    how="left", validate="many_to_one"
)
x_fault_cols = [f"XSIDE_{c}" for c in fault_cols]
focal_fault = enriched[fault_cols].apply(pd.to_numeric, errors="coerce").astype("float32")
xside_fault = enriched[x_fault_cols].apply(pd.to_numeric, errors="coerce").astype("float32")
xside_fault.columns = fault_cols
fault_delta = focal_fault - xside_fault
fault_absdelta = fault_delta.abs()
fault_delta.columns = [f"DELTA_{c}" for c in fault_cols]
fault_absdelta.columns = [f"ABSDELTA_{c}" for c in fault_cols]
enriched = pd.concat([enriched, fault_delta, fault_absdelta], axis=1)

fault_history_audit = {
    "tr_fault_code_count": len(TR_FAULT_CODES),
    "focal_fault_features": len(fault_cols),
    "cross_side_fault_features": len(x_fault_cols),
    "fault_delta_features": len(fault_delta.columns),
    "fault_absdelta_features": len(fault_absdelta.columns),
    "rows_with_any_focal_fault_history": int(enriched[fault_cols].notna().any(axis=1).sum()),
    "rows_with_any_cross_side_fault_history": int(enriched[x_fault_cols].notna().any(axis=1).sum()),
}
(OUTPUT_ROOT / "fault_history_audit.json").write_text(json.dumps(fault_history_audit, indent=2), encoding="utf-8")
print(fault_history_audit)
del fault_all, fault_lookup, focal_fault, xside_fault, fault_delta, fault_absdelta


## Phase 3D: Detailed V3.1 memory plus compact all-code memory

The four detailed families retain nine features each. Every individual fault code in the resolved TR code set receives four compact focal features. Only weighted memory is mirrored cross-side, and only its absolute asymmetry is added, limiting width while testing every code.


In [ ]:
# =============================================================================
# PHASE 3D: detailed family memory + compact all-code landing memory
# =============================================================================
if not CANONICAL_FAULT_FILE.is_file():
    raise FileNotFoundError(CANONICAL_FAULT_FILE)
canonical_faults = pd.read_parquet(CANONICAL_FAULT_FILE, columns=["fault_code"])
ALL_FAULT_CODES = sorted(set(pd.to_numeric(canonical_faults["fault_code"], errors="coerce").dropna().astype(int)) - STARTUP_TRANSIENT_CODES)
if not ALL_FAULT_CODES:
    raise RuntimeError("No usable fault codes resolved")
DETAILED_FAMILIES = {"FAULT204": {204}, "FAULT205": {205}, "FAULT206": {206}, "ANY_TR": set(TR_FAULT_CODES)}

# Separate versioned cache because VER04 V3.1 files do not contain LDGCODE columns.
ALL_CODE_CACHE_ROOT = PART_B_RUN_ROOT / "_shared_feature_cache" / "HYBRID_V3_2_ALL_FAULT_LANDING_MEMORY_VER04"
ALL_CODE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)


def current_streak(history):
    value = 0
    for item in history[::-1]:
        if item: value += 1
        else: break
    return value


def landings_since_last(history):
    positions = np.flatnonzero(history)
    return int(len(history) - positions[-1]) if len(positions) else np.nan


def clean_streak_after_fault(history):
    positions = np.flatnonzero(history)
    return int(len(history) - 1 - positions[-1]) if len(positions) else np.nan


def detailed_memory(presence):
    n=len(presence); keys=["prev1","last3","last5","weighted","streak","since","clean","reappear","accel"]
    out={k:np.zeros(n,dtype=np.float32) for k in keys}; out["since"][:]=np.nan; out["clean"][:]=np.nan
    for i in range(n):
        history=presence[:i]; lag=history[-LANDING_MEMORY_LAGS:][::-1]; padded=np.zeros(LANDING_MEMORY_LAGS,dtype=np.float32); padded[:len(lag)]=lag
        out["prev1"][i]=padded[0]; out["last3"][i]=padded[:3].sum(); out["last5"][i]=padded.sum(); out["weighted"][i]=np.dot(padded,LANDING_MEMORY_WEIGHTS)
        out["streak"][i]=current_streak(history); out["since"][i]=landings_since_last(history); out["clean"][i]=clean_streak_after_fault(history)
        if len(history)>=RECOVERY_CLEAN_LANDINGS+2 and history[-1]==1:
            earlier=np.flatnonzero(history[:-1])
            if len(earlier): out["reappear"][i]=int(len(history)-1-earlier[-1]-1>=RECOVERY_CLEAN_LANDINGS)
        out["accel"][i]=padded[:2].mean()-padded[2:5].mean()
    return out


def compact_memory_matrix(bin_masks, n):
    """Returns prev1, last5, weighted, since for C fault-code rows x N landings."""
    c=bin_masks.shape[0]
    prev=np.zeros((c,n),dtype=np.float32); last5=np.zeros((c,n),dtype=np.float32); weighted=np.zeros((c,n),dtype=np.float32); since=np.full((c,n),np.nan,dtype=np.float32)
    # Shifted presence guarantees current bin exclusion.
    shifted=np.zeros_like(bin_masks,dtype=np.float32)
    if n>1: shifted[:,1:]=bin_masks[:,:-1]
    prev[:]=shifted
    for lag,weight in enumerate(LANDING_MEMORY_WEIGHTS, start=1):
        if lag<n:
            source=bin_masks[:,:n-lag]
            last5[:,lag:]+=source
            weighted[:,lag:]+=source*np.float32(weight)
    last_index=np.full(c,-1,dtype=np.int32)
    for i in range(n):
        since[:,i]=np.where(last_index>=0, i-last_index, np.nan)
        present=np.flatnonzero(bin_masks[:,i])
        if len(present): last_index[present]=i
    return prev,last5,weighted,since

# Synthetic semantics checks.
t=detailed_memory(np.array([0,1,1,0,0,0,1],dtype=np.int8))
assert t["prev1"][2]==1 and t["streak"][3]==2 and t["since"][6]==4 and t["clean"][6]==3
m=np.array([[1,0,0,0,0,0,0]],dtype=np.int8); p,l,w,ss=compact_memory_matrix(m,7)
assert p[0,0]==0 and p[0,1]==1 and np.isclose(w[0,5],.2) and np.isclose(w[0,6],0) and ss[0,1]==1
print("PASS: detailed and all-code compact-memory semantics")


def compute_engine_all_code_memory(args):
    engine, landing_ids, event_ids, cutoff_ns, engine_root_text, cache_root_text, all_codes = args
    path=Path(cache_root_text)/f"engine={engine}.parquet"; result={"engine_serial":engine,"status":"FAIL","rows":0,"error":""}
    try:
        order=np.lexsort((landing_ids.astype(str),cutoff_ns)); row_ids=landing_ids[order].astype(str); event_ids=event_ids[order].astype(str); cutoff_ns=cutoff_ns[order]
        unique=(pd.DataFrame({"event_id":event_ids,"cutoff_ns":cutoff_ns}).sort_values(["cutoff_ns","event_id"]).drop_duplicates("event_id").reset_index(drop=True))
        cutoffs=unique.cutoff_ns.to_numpy(dtype=np.int64); unique_ids=unique.event_id.astype(str).to_numpy(); n=len(cutoffs)
        event_ns=np.array([],dtype=np.int64); event_codes=np.array([],dtype=np.int64)
        hp=Path(engine_root_text)/f"engine={engine}"/"history_exact_deduplicated.parquet"
        if hp.is_file() and {"record_datetime_utc","fault_code"}.issubset(set(pq.ParquetFile(hp).schema_arrow.names)):
            h=pd.read_parquet(hp,columns=["record_datetime_utc","fault_code"]); ts=pd.to_datetime(h.record_datetime_utc,errors="coerce",utc=True); fc=pd.to_numeric(h.fault_code,errors="coerce"); valid=ts.notna()&fc.notna()
            event_ns=ts[valid].astype("int64").to_numpy(dtype=np.int64); event_codes=fc[valid].astype(int).to_numpy(dtype=np.int64)
            if len(event_ns):
                pair=np.rec.fromarrays([event_ns,event_codes],names="t,c");_,keep=np.unique(pair,return_index=True);event_ns,event_codes=event_ns[keep],event_codes[keep]
        bins=np.searchsorted(cutoffs,event_ns,side="left") if len(event_ns) else np.array([],dtype=np.int64); valid=bins<n; bins,event_codes=bins[valid],event_codes[valid]
        code_to_row={code:i for i,code in enumerate(all_codes)}; masks=np.zeros((len(all_codes),n),dtype=np.int8)
        for b,code in zip(bins,event_codes):
            r=code_to_row.get(int(code))
            if r is not None:masks[r,b]=1
        prev,last5,weighted,since=compact_memory_matrix(masks,n)
        data={"aircraft_landing_event_id":unique_ids}
        # Detailed known families.
        suffix={"prev1":"PREV1_PRESENT","last3":"LAST3_COUNT","last5":"LAST5_COUNT","weighted":"WEIGHTED_LAST5","streak":"PRIOR_STREAK","since":"LANDINGS_SINCE_LAST","clean":"CLEAN_STREAK_AFTER_LAST","reappear":"REAPPEARED_AFTER_CLEAN","accel":"RECENT_ACCELERATION"}
        for family,codes in DETAILED_FAMILIES.items():
            presence=np.zeros(n,dtype=np.int8); selected=bins[np.isin(event_codes,list(codes))]
            if len(selected):presence[np.unique(selected)]=1
            for key,values in detailed_memory(presence).items():data[f"LDGMEM_{family}_{suffix[key]}"]=values
        for r,code in enumerate(all_codes):
            token=f"{int(code):03d}"
            data[f"LDGCODE_{token}_PREV1_PRESENT"]=prev[r]
            data[f"LDGCODE_{token}_LAST5_COUNT"]=last5[r]
            data[f"LDGCODE_{token}_WEIGHTED_LAST5"]=weighted[r]
            data[f"LDGCODE_{token}_LANDINGS_SINCE_LAST"]=since[r]
        features=pd.DataFrame(data)
        output=pd.DataFrame({"engine_channel_landing_id":row_ids,"aircraft_landing_event_id":event_ids}).merge(features,on="aircraft_landing_event_id",how="left",validate="many_to_one").drop(columns="aircraft_landing_event_id")
        for c in output.columns:
            if c!="engine_channel_landing_id":output[c]=pd.to_numeric(output[c],errors="coerce").astype("float32")
        output.to_parquet(path,index=False,compression="zstd"); result.update(status="PASS",rows=len(output),columns=len(output.columns))
    except Exception:result["error"]=traceback.format_exc(limit=8)
    return result

memory_tasks=[]
for engine,rows in enriched.groupby(enriched.engine_serial.astype(str),sort=False):
    marker=ALL_CODE_CACHE_ROOT/f"engine={engine}.parquet"; valid=False
    if marker.is_file():
        try:
            pf=pq.ParquetFile(marker); names=set(pf.schema_arrow.names); valid=(pf.metadata.num_rows==len(rows) and "engine_channel_landing_id" in names and any(c.startswith("LDGCODE_") for c in names))
        except Exception:valid=False
    if valid:continue
    marker.unlink(missing_ok=True)
    memory_tasks.append((str(engine),rows.engine_channel_landing_id.astype(str).to_numpy(),rows.aircraft_landing_event_id.astype(str).to_numpy(),rows.feature_cutoff_utc.astype("int64").to_numpy(dtype=np.int64),str(ENGINE_ROOT),str(ALL_CODE_CACHE_ROOT),ALL_FAULT_CODES))
print({"engines":enriched.engine_serial.nunique(),"cached":enriched.engine_serial.nunique()-len(memory_tasks),"remaining":len(memory_tasks),"workers":LANDING_MEMORY_WORKERS,"fault_codes":len(ALL_FAULT_CODES)})
results=[]
if memory_tasks:
    with ProcessPoolExecutor(max_workers=LANDING_MEMORY_WORKERS) as executor:
        futures={executor.submit(compute_engine_all_code_memory,t):t[0] for t in memory_tasks}
        for i,future in enumerate(as_completed(futures),1):
            engine=futures[future]
            try:r=future.result()
            except Exception:r={"engine_serial":engine,"status":"FAIL","error":traceback.format_exc(limit=8)}
            results.append(r)
            if i==1 or i%100==0 or i==len(memory_tasks):print(f"[V3.2 {i:,}/{len(memory_tasks):,}] {engine}: {r['status']}")
failed=[r for r in results if r["status"]!="PASS"]
if failed:
    pd.DataFrame(failed).to_csv(OUTPUT_ROOT/"all_code_memory_failures_ver04.csv",index=False);raise RuntimeError(f"{len(failed)} engine failures")
files=sorted(ALL_CODE_CACHE_ROOT.glob("engine=*.parquet")); memory_all=pd.concat([pd.read_parquet(p) for p in files],ignore_index=True,sort=False);memory_all.engine_channel_landing_id=memory_all.engine_channel_landing_id.astype(str)
if memory_all.engine_channel_landing_id.duplicated().any():raise RuntimeError("duplicate landing IDs in V3.2 cache")
enriched.engine_channel_landing_id=enriched.engine_channel_landing_id.astype(str);before=len(enriched);enriched=enriched.merge(memory_all,on="engine_channel_landing_id",how="left",validate="one_to_one")
if len(enriched)!=before:raise RuntimeError("V3.2 join changed row count")
# Recreate the complete V3.1 detailed-family parent: focal, cross-side,
# signed delta, and absolute delta. This is required for a clean V3.1 replay.
detailed_cols=[c for c in enriched.columns if c.startswith("LDGMEM_")]
detailed_lookup=(enriched[["aircraft_landing_event_id","engine_serial"]+detailed_cols]
    .drop_duplicates(["aircraft_landing_event_id","engine_serial"])
    .rename(columns={"engine_serial":"cross_side_engine_serial",**{c:f"XSIDE_{c}" for c in detailed_cols}}))
enriched=enriched.merge(detailed_lookup,on=["aircraft_landing_event_id","cross_side_engine_serial"],how="left",validate="many_to_one")
xdetailed=[f"XSIDE_{c}" for c in detailed_cols]
focal_d=enriched[detailed_cols].astype("float32"); xside_d=enriched[xdetailed].astype("float32"); xside_d.columns=detailed_cols
delta_d=focal_d-xside_d; absdelta_d=delta_d.abs()
delta_d.columns=[f"DELTA_{c}" for c in detailed_cols]; absdelta_d.columns=[f"ABSDELTA_{c}" for c in detailed_cols]
enriched=pd.concat([enriched,delta_d,absdelta_d],axis=1)

# Cross-side only for weighted per-code memory, then absolute asymmetry only.
allcode_cols=[c for c in enriched.columns if c.startswith("LDGCODE_")]
weighted_cols=[c for c in allcode_cols if c.endswith("_WEIGHTED_LAST5")]
lookup=(enriched[["aircraft_landing_event_id","engine_serial"]+weighted_cols].drop_duplicates(["aircraft_landing_event_id","engine_serial"]).rename(columns={"engine_serial":"cross_side_engine_serial",**{c:f"XSIDE_{c}" for c in weighted_cols}}))
enriched=enriched.merge(lookup,on=["aircraft_landing_event_id","cross_side_engine_serial"],how="left",validate="many_to_one")
xweighted=[f"XSIDE_{c}" for c in weighted_cols]
focal=enriched[weighted_cols].astype("float32");xside=enriched[xweighted].astype("float32");xside.columns=weighted_cols
absdelta=(focal-xside).abs();absdelta.columns=[f"ABSDELTA_{c}" for c in weighted_cols];enriched=pd.concat([enriched,absdelta],axis=1)
# Support audit by code.
support=[]
for code in ALL_FAULT_CODES:
    token=f"{int(code):03d}"; prev=f"LDGCODE_{token}_PREV1_PRESENT"; last5=f"LDGCODE_{token}_LAST5_COUNT"
    support.append({"fault_code":int(code),"prev1_positive_rows":int((enriched[prev]>0).sum()),"last5_positive_rows":int((enriched[last5]>0).sum()),"eligible_support":bool((enriched[prev]>0).sum()>=100 or (enriched[last5]>0).sum()>=100)})
support_frame=pd.DataFrame(support)
support_frame.to_csv(OUTPUT_ROOT/"all_fault_code_support_ver04.csv",index=False)
ELIGIBLE_ALL_FAULT_CODES=set(support_frame.loc[support_frame["eligible_support"],"fault_code"].astype(int))
ELIGIBLE_ALL_CODE_TOKENS={f"{code:03d}" for code in ELIGIBLE_ALL_FAULT_CODES}
print({"resolved_fault_codes":len(ALL_FAULT_CODES),"eligible_fault_codes":len(ELIGIBLE_ALL_FAULT_CODES)})
all_code_audit={"contract":"HYBRID_V3_2_ALL_FAULT_LANDING_MEMORY_VER04","fault_codes":len(ALL_FAULT_CODES),"focal_all_code_features":len(allcode_cols),"cross_side_weighted_features":len(xweighted),"absolute_weighted_delta_features":len(absdelta.columns),"detailed_v31_focal_features":len(detailed_cols),"detailed_v31_cross_side_features":len(xdetailed),"detailed_v31_delta_features":len(delta_d.columns),"detailed_v31_absdelta_features":len(absdelta_d.columns),"workers":LANDING_MEMORY_WORKERS,"current_bin_excluded":True}
(OUTPUT_ROOT/"all_code_memory_audit_ver04.json").write_text(json.dumps(all_code_audit,indent=2),encoding="utf-8");print(all_code_audit)
del canonical_faults,memory_all,detailed_lookup,focal_d,xside_d,delta_d,absdelta_d,lookup,focal,xside,absdelta


## Phase 3D: Compact 8004-informed operating-model features

This phase reads each exact engine history independently and writes only one narrow landing-keyed feature file per engine. It does not persist cycles, transitions, cross-side copies, deltas, or an enriched full matrix. The compact cache is reusable across notebook restarts and separate benchmark runs.

Feature scope is intentionally constrained to cycle completion, interval-censored response timing, sample quality, retries/toggles, decoded TR flag behavior, controller FAILED transitions/dwell, robust engine-model deviation, and same-event cross-side asymmetry.


In [ ]:
# =============================================================================
# PHASE 3D: 8004-informed operating-model features
# =============================================================================
STATE_FIELDS = [
    "sw3_tr_controller_state_code", "sw3_tr_deploy_command", "sw3_tr_stow_command",
    "sw3_tr_deployed", "sw3_tr_stowed", "sw3_tr_position_indeterminate",
    "sw3_tr_dcu_solenoid_command", "sw3_tr_isolation_solenoid_command",
    "sw3_tr_isolation_pressure_switch", "sw3_tr_unlock_1plu", "sw3_tr_unlock_2plu",
    "sw3_tr_prime_lock_balk_switch", "sw3_tr_hydraulic_control_1", "sw3_tr_hydraulic_control_2",
    "sw3_tr_state_error", "sw3_tr_solenoid_fault", "sw3_tr_position_resistor_fault", "sw3_tr_unsafe",
]
FLAG_TERMS = ("fault", "unsafe", "state_error", "failure", "failed", "disagree", "invalid", "indeterminate")
EXPECTED_TRANSITIONS = {("CLOSED","OPENING"), ("OPENING","OPEN"), ("OPEN","CLOSING"), ("CLOSING","CLOSED")}

def rising(series):
    x = pd.to_numeric(series, errors="coerce").ffill().fillna(0)
    return x.eq(1) & x.shift(fill_value=0).eq(0)

def build_cycles_8004(g, direction, command_col, target_col):
    rows=[]
    command_positions=np.flatnonzero(rising(g[command_col]).to_numpy())
    for n,pos in enumerate(command_positions):
        t0=float(g.iloc[pos]["ecu_time"])
        next_pos=int(command_positions[n+1]) if n+1<len(command_positions) else len(g)
        stop_time=t0+MAX_CYCLE_ECU
        later=g.iloc[pos:next_pos]
        hit=later.index[(pd.to_numeric(later[target_col],errors="coerce")==1) & (later["ecu_time"]<=stop_time)]
        target_pos=int(hit[0]) if len(hit) else None
        t1=float(g.loc[target_pos,"ecu_time"]) if target_pos is not None else np.nan
        response=t1-t0 if pd.notna(t1) else np.nan
        stop=target_pos if target_pos is not None else int(g.index[min(len(g)-1,pos+int(MAX_CYCLE_ECU)+2)])
        segment=g.loc[pos:stop]
        sample_gaps=np.diff(segment["ecu_time"].to_numpy(dtype=float)) if len(segment)>1 else np.array([])
        quality=("INCOMPLETE" if target_pos is None else
                 "SAME_RECORD_INTERVAL_CENSORED" if response==0 else
                 "POSITIVE_INTERVAL_CENSORED" if response<=MAX_CYCLE_ECU else "ABOVE_QA_MAX")
        row={
            "direction":direction,"command_ecu":t0,"target_ecu":t1,"response_observed":response,
            "response_lower":max(0,response-NOMINAL_SAMPLE_ECU) if pd.notna(response) else np.nan,
            "response_upper":response+NOMINAL_SAMPLE_ECU if pd.notna(response) else np.nan,
            "response_quality":quality,"complete":int(target_pos is not None),
            "same_record":int(response==0 if pd.notna(response) else False),
            "max_internal_sample_gap":float(sample_gaps.max()) if len(sample_gaps) else np.nan,
            "cycle_calendar_time":g.iloc[target_pos]["record_datetime_utc"] if target_pos is not None else g.iloc[pos]["record_datetime_utc"],
        }
        # Flag behavior and toggles within the observed cycle.
        flag_cols=[c for c in segment.columns if c.startswith("sw3_tr_") and any(t in c.lower() for t in FLAG_TERMS)]
        row["flag_active_count"]=sum(int(pd.to_numeric(segment[c],errors="coerce").eq(1).any()) for c in flag_cols)
        row["flag_rise_count"]=sum(int(rising(segment[c]).sum()) for c in flag_cols)
        behavior_cols=[c for c in STATE_FIELDS if c in segment.columns]
        row["status_toggle_count"]=sum(int(pd.to_numeric(segment[c],errors="coerce").ffill().diff().ne(0).sum()) for c in behavior_cols)
        row["command_retry_count"]=max(0,int(rising(segment[command_col]).sum())-1)
        rows.append(row)
    return rows

def controller_features(g):
    """Observed controller transitions with dwell from state entry to exit."""
    rows=[]; state_col="sw3_tr_controller_state_code"
    if state_col not in g.columns:return rows
    state=pd.to_numeric(g[state_col],errors="coerce").ffill()
    if state.notna().sum()<2:return rows
    entry_index=0
    for i in range(1,len(g)):
        a=state.iloc[i-1]; b=state.iloc[i]
        if pd.isna(a) or pd.isna(b):
            entry_index=i; continue
        if a!=b:
            af=STATE_NAMES.get(int(a),str(int(a))); bt=STATE_NAMES.get(int(b),str(int(b)))
            dwell=float(g["ecu_time"].iloc[i]-g["ecu_time"].iloc[entry_index])
            rows.append({"from_state":af,"to_state":bt,"interval":dwell,
                         "illegal":int((af,bt) not in EXPECTED_TRANSITIONS and bt=="FAILED"),
                         "transition_calendar_time":g["record_datetime_utc"].iloc[i]})
            entry_index=i
    return rows

# Verify FAILED dwell is measured from state entry, not merely adjacent sample gap.
_controller_test=pd.DataFrame({
    "ecu_time":[0.,1.,2.,3.,4.,5.],
    "record_datetime_utc":pd.date_range("2020-01-01",periods=6,freq="s",tz="UTC"),
    "sw3_tr_controller_state_code":[0,5,5,5,5,0],
})
_controller_rows=controller_features(_controller_test)
assert any(r["from_state"]=="FAILED" and np.isclose(r["interval"],4.0) for r in _controller_rows)
print("PASS: controller dwell semantics")

def extract_engine_8004(args):
    engine, landing_ids, cutoff_ns, engine_root_text, cache_root_text = args
    out_path=Path(cache_root_text)/f"engine={engine}.parquet"
    result={"engine_serial":engine,"status":"FAIL","rows":0,"error":""}
    try:
        hist_path=Path(engine_root_text)/f"engine={engine}"/"history_exact_deduplicated.parquet"
        names=pq.ParquetFile(hist_path).schema_arrow.names
        base=[c for c in ["record_datetime_utc","ecu_operating_time_raw","leg_number","ecu_channel","resolved_engine_model"] if c in names]
        dynamic_flags = [c for c in names if c.startswith("sw3_tr_") and any(t in c.lower() for t in FLAG_TERMS)]
        fields = sorted(set([c for c in STATE_FIELDS if c in names] + dynamic_flags))
        required={"record_datetime_utc","ecu_operating_time_raw","leg_number","ecu_channel","sw3_tr_deploy_command","sw3_tr_stow_command","sw3_tr_deployed","sw3_tr_stowed"}
        output=pd.DataFrame({"engine_channel_landing_id":landing_ids})
        if required.issubset(set(names)):
            hist=pd.read_parquet(hist_path,columns=list(dict.fromkeys(base+fields)))
            hist["record_datetime_utc"]=pd.to_datetime(hist["record_datetime_utc"],errors="coerce",utc=True)
            hist["ecu_time"]=pd.to_numeric(hist["ecu_operating_time_raw"],errors="coerce")
            hist["leg"]=pd.to_numeric(hist["leg_number"],errors="coerce")
            hist=hist.loc[hist["record_datetime_utc"].notna()&hist["ecu_time"].notna()&hist["leg"].notna()].copy()
            model=(hist["resolved_engine_model"].dropna().astype(str).mode().iloc[0]
                   if "resolved_engine_model" in hist and hist["resolved_engine_model"].notna().any() else "UNKNOWN_MODEL")
            cycles=[];transitions=[]
            for _,g in hist.groupby(["leg","ecu_channel"],dropna=False,sort=False):
                g=g.sort_values(["ecu_time","record_datetime_utc"]).drop_duplicates("ecu_time",keep="last").reset_index(drop=True)
                cycles += build_cycles_8004(g,"DEPLOY","sw3_tr_deploy_command","sw3_tr_deployed")
                cycles += build_cycles_8004(g,"STOW","sw3_tr_stow_command","sw3_tr_stowed")
                transitions += controller_features(g)
            cyc=pd.DataFrame(cycles);trn=pd.DataFrame(transitions)
            # Robust engine-model baseline is computed from this engine's observed cycles here;
            # population model normalization is added after all engines are consolidated.
            feature_dict={}
            cycle_time=(pd.to_datetime(cyc["cycle_calendar_time"],utc=True).astype("int64").to_numpy(dtype=np.int64) if len(cyc) else np.array([],dtype=np.int64))
            transition_time=(pd.to_datetime(trn["transition_calendar_time"],utc=True).astype("int64").to_numpy(dtype=np.int64) if len(trn) else np.array([],dtype=np.int64))
            for days in STATE_LOOKBACK_DAYS:
                width=np.int64(pd.Timedelta(days=days).value)
                for direction in ["DEPLOY","STOW"]:
                    sub=cyc.loc[cyc.direction.eq(direction)].copy() if len(cyc) else pd.DataFrame()
                    st=(pd.to_datetime(sub.cycle_calendar_time,utc=True).astype("int64").to_numpy(dtype=np.int64) if len(sub) else np.array([],dtype=np.int64))
                    for metric in ["attempts","completion_rate","same_record_rate","incomplete_rate","response_median","response_p90",
                                   "flag_active_mean","flag_rise_mean","toggle_mean","retry_mean","sample_gap_mean"]:
                        feature_dict[f"OPMOD_{direction}_{metric.upper()}_{days}D"]=np.full(len(cutoff_ns),np.nan,dtype=np.float32)
                    for i,anchor in enumerate(cutoff_ns):
                        left=np.searchsorted(st,anchor-width,side="right");right=np.searchsorted(st,anchor,side="right")
                        z=sub.iloc[left:right]
                        feature_dict[f"OPMOD_{direction}_ATTEMPTS_{days}D"][i]=len(z)
                        if len(z):
                            complete=z.complete.astype(float)
                            feature_dict[f"OPMOD_{direction}_COMPLETION_RATE_{days}D"][i]=complete.mean()
                            feature_dict[f"OPMOD_{direction}_SAME_RECORD_RATE_{days}D"][i]=z.same_record.mean()
                            feature_dict[f"OPMOD_{direction}_INCOMPLETE_RATE_{days}D"][i]=1-complete.mean()
                            rv=pd.to_numeric(z.response_observed,errors="coerce").dropna()
                            if len(rv):
                                feature_dict[f"OPMOD_{direction}_RESPONSE_MEDIAN_{days}D"][i]=rv.median()
                                feature_dict[f"OPMOD_{direction}_RESPONSE_P90_{days}D"][i]=rv.quantile(.9)
                            for source,name in [("flag_active_count","FLAG_ACTIVE_MEAN"),("flag_rise_count","FLAG_RISE_MEAN"),
                                                ("status_toggle_count","TOGGLE_MEAN"),("command_retry_count","RETRY_MEAN"),
                                                ("max_internal_sample_gap","SAMPLE_GAP_MEAN")]:
                                feature_dict[f"OPMOD_{direction}_{name}_{days}D"][i]=pd.to_numeric(z[source],errors="coerce").mean()
                for name in ["TRANSITIONS","FAILED_TRANSITIONS","FAILED_TRANSITION_RATE","FAILED_DWELL_MEAN","FAILED_DWELL_MAX"]:
                    feature_dict[f"OPMOD_CONTROLLER_{name}_{days}D"]=np.full(len(cutoff_ns),np.nan,dtype=np.float32)
                for i,anchor in enumerate(cutoff_ns):
                    left=np.searchsorted(transition_time,anchor-width,side="right");right=np.searchsorted(transition_time,anchor,side="right")
                    z=trn.iloc[left:right] if len(trn) else pd.DataFrame()
                    feature_dict[f"OPMOD_CONTROLLER_TRANSITIONS_{days}D"][i]=len(z)
                    if len(z):
                        failed=z.to_state.eq("FAILED")
                        feature_dict[f"OPMOD_CONTROLLER_FAILED_TRANSITIONS_{days}D"][i]=failed.sum()
                        feature_dict[f"OPMOD_CONTROLLER_FAILED_TRANSITION_RATE_{days}D"][i]=failed.mean()
                        dwell=pd.to_numeric(z.loc[z.from_state.eq("FAILED"),"interval"],errors="coerce").dropna()
                        if len(dwell):
                            feature_dict[f"OPMOD_CONTROLLER_FAILED_DWELL_MEAN_{days}D"][i]=dwell.mean()
                            feature_dict[f"OPMOD_CONTROLLER_FAILED_DWELL_MAX_{days}D"][i]=dwell.max()
            output=pd.DataFrame({"engine_channel_landing_id":landing_ids,"OPMOD_ENGINE_MODEL":model,**feature_dict})
        # Narrow persisted numeric types. The engine model remains categorical text.
        for column in output.columns:
            if column in {"engine_channel_landing_id", "OPMOD_ENGINE_MODEL"}:
                continue
            output[column] = pd.to_numeric(output[column], errors="coerce").astype("float32")
        output.to_parquet(out_path,index=False,compression="zstd")
        result.update(status="PASS",rows=len(output))
    except Exception:
        result["error"]=traceback.format_exc(limit=8)
    return result

# Disk and cache contract guards. This cache contains compact features only.
free_gib = shutil.disk_usage(STATE_CACHE_ROOT).free / (1024 ** 3)
if free_gib < MIN_FREE_GIB:
    raise RuntimeError(f"Only {free_gib:.1f} GiB free; require at least {MIN_FREE_GIB:.1f} GiB")

required_cache_columns = {"engine_channel_landing_id", "OPMOD_ENGINE_MODEL"}
valid_cache_files = []
invalid_cache_files = []
for cache_file in STATE_CACHE_ROOT.glob("engine=*.parquet"):
    try:
        pf_cache = pq.ParquetFile(cache_file)
        if pf_cache.metadata.num_rows > 0 and required_cache_columns.issubset(set(pf_cache.schema_arrow.names)):
            valid_cache_files.append(cache_file)
        else:
            invalid_cache_files.append(cache_file)
    except Exception:
        invalid_cache_files.append(cache_file)
for cache_file in invalid_cache_files:
    cache_file.unlink(missing_ok=True)
print({"valid_compact_engine_caches": len(valid_cache_files), "removed_invalid_caches": len(invalid_cache_files), "free_gib": round(free_gib, 2)})

state_tasks=[]
for engine,rows in enriched.groupby(enriched["engine_serial"].astype(str),sort=False):
    marker=STATE_CACHE_ROOT/f"engine={engine}.parquet"
    if marker.is_file():continue
    state_tasks.append((str(engine),rows["engine_channel_landing_id"].astype(str).to_numpy(),
                        rows["feature_cutoff_utc"].astype("int64").to_numpy(dtype=np.int64),str(ENGINE_ROOT),str(STATE_CACHE_ROOT)))
print({"state_model_engines_total":enriched["engine_serial"].nunique(),"remaining":len(state_tasks)})
state_results=[]
if state_tasks:
    with ProcessPoolExecutor(max_workers=STATE_WORKERS) as executor:
        futures={executor.submit(extract_engine_8004,t):t[0] for t in state_tasks}
        for i,future in enumerate(as_completed(futures),1):
            engine=futures[future]
            try:r=future.result()
            except Exception:r={"engine_serial":engine,"status":"FAIL","error":traceback.format_exc(limit=8)}
            state_results.append(r)
            if i==1 or i%100==0 or i==len(state_tasks):print(f"[OPMOD {i:,}/{len(state_tasks):,}] {engine}: {r['status']}")
if state_results:
    failed=[r for r in state_results if r["status"]!="PASS"]
    if failed:
        pd.DataFrame(failed).to_csv(OUTPUT_ROOT/"state_model_failures.csv",index=False)
        raise RuntimeError(f"{len(failed)} engines failed operating-model generation")

state_files=sorted(STATE_CACHE_ROOT.glob("engine=*.parquet"))
if not state_files:
    raise RuntimeError("No compact V4B engine caches were found")
cache_manifest_rows = []
cache_bytes = 0
for path in state_files:
    pf_cache = pq.ParquetFile(path)
    size = path.stat().st_size
    cache_bytes += size
    cache_manifest_rows.append({
        "engine_serial": path.stem.split("=", 1)[-1],
        "path": str(path),
        "rows": int(pf_cache.metadata.num_rows),
        "columns": len(pf_cache.schema_arrow.names),
        "size_bytes": size,
        "contract": STATE_CACHE_CONTRACT,
    })
cache_gib = cache_bytes / (1024 ** 3)
if cache_gib > MAX_STATE_CACHE_GIB:
    raise RuntimeError(f"Compact V4B cache reached {cache_gib:.2f} GiB, above budget {MAX_STATE_CACHE_GIB:.2f} GiB")
pd.DataFrame(cache_manifest_rows).to_parquet(STATE_CACHE_MANIFEST,index=False,compression="zstd")
print({"compact_cache_files":len(state_files),"compact_cache_gib":round(cache_gib,3),"cache_manifest":str(STATE_CACHE_MANIFEST)})
state_all=pd.concat([pd.read_parquet(p) for p in state_files],ignore_index=True,sort=False)
state_all["engine_channel_landing_id"]=state_all["engine_channel_landing_id"].astype(str)
enriched=enriched.merge(state_all,on="engine_channel_landing_id",how="left",validate="one_to_one")
op_cols=[c for c in enriched.columns if c.startswith("OPMOD_") and c!="OPMOD_ENGINE_MODEL"]

# Population model-direction normalization: robust median/IQR by engine model.
# This is label-free. The notebook exports the baselines for audit.
baseline_rows=[]
for model,g in enriched.groupby("OPMOD_ENGINE_MODEL",dropna=False):
    for c in [x for x in op_cols if "RESPONSE_" in x or "COMPLETION_RATE" in x or "FAILED_TRANSITION_RATE" in x]:
        x=pd.to_numeric(g[c],errors="coerce").dropna()
        if len(x)<100:continue
        q25,q50,q75=x.quantile([.25,.5,.75]);iqr=max(float(q75-q25),1e-6)
        baseline_rows.append({"engine_model":model,"feature":c,"median":float(q50),"iqr":iqr,"rows":len(x)})
baselines=pd.DataFrame(baseline_rows)
baselines.to_csv(OUTPUT_ROOT/"engine_model_robust_baselines.csv",index=False)
for r in baselines.itertuples(index=False):
    mask=enriched["OPMOD_ENGINE_MODEL"].eq(r.engine_model)
    enriched.loc[mask,f"{r.feature}__MODEL_RZ"]=(pd.to_numeric(enriched.loc[mask,r.feature],errors="coerce")-r.median)/r.iqr
op_cols=[c for c in enriched.columns if c.startswith("OPMOD_") and c!="OPMOD_ENGINE_MODEL"]

# Same-aircraft-event opposite-engine operating-model values and deltas.
op_lookup=enriched[["aircraft_landing_event_id","engine_serial"]+op_cols].drop_duplicates(
    ["aircraft_landing_event_id","engine_serial"],keep="first").rename(
    columns={"engine_serial":"cross_side_engine_serial",**{c:f"XSIDE_{c}" for c in op_cols}})
enriched=enriched.merge(op_lookup,on=["aircraft_landing_event_id","cross_side_engine_serial"],how="left",validate="many_to_one")
x_op_cols=[f"XSIDE_{c}" for c in op_cols]
focal=enriched[op_cols].apply(pd.to_numeric,errors="coerce").astype("float32")
xside=enriched[x_op_cols].apply(pd.to_numeric,errors="coerce").astype("float32");xside.columns=op_cols
delta=focal-xside;absdelta=delta.abs();delta.columns=[f"DELTA_{c}" for c in op_cols];absdelta.columns=[f"ABSDELTA_{c}" for c in op_cols]
enriched=pd.concat([enriched,delta,absdelta],axis=1)
operating_model_audit={
    "focal_operating_model_features":len(op_cols),"cross_side_features":len(x_op_cols),
    "delta_features":len(delta.columns),"absolute_delta_features":len(absdelta.columns),
    "engine_models":int(enriched["OPMOD_ENGINE_MODEL"].nunique(dropna=True)),
    "robust_model_baselines":len(baselines),
    "rows_with_any_focal_operating_model":int(enriched[op_cols].notna().any(axis=1).sum()),
    "rows_with_any_cross_side_operating_model":int(enriched[x_op_cols].notna().any(axis=1).sum()),
    "timing_basis":"ECU operating time within cycle; calendar time only for cycle-to-landing lookback",
    "nominal_sample_ecu":NOMINAL_SAMPLE_ECU,"max_cycle_ecu":MAX_CYCLE_ECU,
    "compact_cache_files":len(state_files),"compact_cache_gib":cache_gib,
    "full_model_input_rewritten":False,"enriched_matrix_persisted":False,
    "cross_side_and_delta_columns_persisted":False,
}
(OUTPUT_ROOT/"operating_model_audit.json").write_text(json.dumps(operating_model_audit,indent=2),encoding="utf-8")
print(operating_model_audit)
del state_all,op_lookup,focal,xside,delta,absdelta


## Phase 4: Rebuild the FF206 target

Same explicit-FF-GMT label construction as the existing CELL 10, left
unchanged so the label definition stays directly comparable -- only the
feature set differs between this run and the baseline.

In [ ]:
# =============================================================================
# PHASE 4: rebuild the FF206 target (same explicit-FF-GMT logic as the
# existing CELL 10 -- unchanged so the label definition stays comparable)
# =============================================================================
parts = sorted(ROW_V2_ROOT.glob("file_hash=*/part-*.parquet"))
selected_engine_set = set(enriched["engine_serial"].astype(str))
ff_frames = []
for index, path in enumerate(parts, 1):
    schema = set(pq.ParquetFile(path).schema_arrow.names)
    if not {"engine_serial", "section", "fault_code", "record_datetime_utc"}.issubset(schema):
        continue
    needed = [c for c in ["engine_serial", "section", "fault_code", "record_datetime_utc"] if c in schema]
    frame = pd.read_parquet(path, columns=needed)
    mask = (
        frame["engine_serial"].astype(str).isin(selected_engine_set)
        & frame["section"].fillna("").astype(str).str.upper().eq("FF_ATTRIBUTES")
        & pd.to_numeric(frame["fault_code"], errors="coerce").eq(206)
    )
    frame = frame.loc[mask].copy()
    if not frame.empty:
        frame["fault206_datetime_utc"] = pd.to_datetime(frame["record_datetime_utc"], errors="coerce", utc=True)
        ff_frames.append(frame)
    if index == 1 or index % 10000 == 0 or index == len(parts):
        print(f"Fault 206 scan {index:,}/{len(parts):,}; matching frames={len(ff_frames):,}")

if not ff_frames:
    raise RuntimeError("No corrected FF_ATTRIBUTES Fault 206 rows found")
fault206 = pd.concat(ff_frames, ignore_index=True, sort=False)
fault206 = fault206.loc[
    fault206["fault206_datetime_utc"].notna()
    & fault206["fault206_datetime_utc"].ge(MIN_TS)
    & fault206["fault206_datetime_utc"].le(MAX_TS)
].drop_duplicates(["engine_serial", "fault206_datetime_utc"])

fault_times_by_engine = {
    str(engine): np.sort(group["fault206_datetime_utc"].astype("int64").to_numpy(dtype=np.int64))
    for engine, group in fault206.groupby("engine_serial", sort=False)
}
cutoff_ns = enriched["feature_cutoff_utc"].astype("int64").to_numpy(dtype=np.int64)
horizon_ns = np.int64(pd.Timedelta(days=HORIZON_DAYS).value)
labels = np.zeros(len(enriched), dtype=np.int8)
for position, (engine, cutoff) in enumerate(zip(enriched["engine_serial"].astype(str), cutoff_ns)):
    times = fault_times_by_engine.get(engine)
    if times is None or len(times) == 0:
        continue
    idx = np.searchsorted(times, cutoff, side="right")
    if idx >= len(times):
        continue
    labels[position] = np.int8(np.int64(times[idx]) <= cutoff + horizon_ns)
TARGET = "FAULT206_WITHIN_14D_AFTER_WINDOW"
enriched[TARGET] = labels

## Phase 5: Isolated V3, V3.2, V4B, and V4B.2 feature screens


In [ ]:
# =============================================================================
# PHASE 5: isolated V3 / V3.2 / V4B / V4B.2 screens
# =============================================================================
identifier_columns={"aircraft_landing_event_id","engine_channel_landing_id","engine_landing_id","engine_serial","aircraft_identifier","aircraft_serial_number","aircraft_registration","cross_side_engine_serial","resolved_engine_model","OPMOD_ENGINE_MODEL","ecu_channel","ecu_channel_norm","landing_datetime_utc","landing_ecu_time","feature_cutoff_utc",TARGET}
blocked_patterns=[r"FAULT__206_COUNT",r"TR_FAULT_206",r"(?<!\d)175(?!\d)",r"(?<!\d)176(?!\d)",r"STARTUP_175_176",r"FUTURE",r"TARGET",r"LABEL",r"NEXT_FAULT"]
def allowed_candidate(c):return c not in identifier_columns and not any(re.search(p,c,re.I) for p in blocked_patterns)
def is_memory(c):return "LDGMEM_" in c or "LDGCODE_" in c
def is_opmodel(c):return "OPMOD_" in c
def all_code_supported(c):
    if "LDGCODE_" not in c:
        return True
    match=re.search(r"LDGCODE_(\d+)_",c)
    return bool(match and match.group(1) in ELIGIBLE_ALL_CODE_TOKENS)
all_candidates=[c for c in enriched.columns if allowed_candidate(c) and all_code_supported(c)]
v3_candidates=[c for c in all_candidates if not is_memory(c) and not is_opmodel(c)]
v32_candidates=[c for c in all_candidates if not is_opmodel(c)]
v4b_candidates=[c for c in all_candidates if not is_memory(c)]
v4b2_candidates=list(all_candidates)
y_full=enriched[TARGET].astype(int)
def score_features(columns,tag):
    scored=[]
    for i,c in enumerate(columns,1):
        x=pd.to_numeric(enriched[c],errors="coerce");n=int(x.notna().sum())
        if n<100 or x.nunique(dropna=True)<=1:continue
        mask=x.notna()
        if y_full[mask].nunique()<2:continue
        corr=np.corrcoef(x[mask],y_full[mask])[0,1];scored.append((c,abs(corr) if np.isfinite(corr) else 0.0,n))
        if i==1 or i%1000==0 or i==len(columns):print(f"[{tag}] {i:,}/{len(columns):,}")
    scored.sort(key=lambda z:z[1],reverse=True)
    pd.DataFrame(scored,columns=["feature","abs_correlation_with_target","nonmissing_rows"]).to_csv(OUTPUT_ROOT/f"feature_ranking_{tag}.csv",index=False)
    return scored
scores={"v3":score_features(v3_candidates,"hybrid_v3"),"v32":score_features(v32_candidates,"hybrid_v3_2"),"v4b":score_features(v4b_candidates,"hybrid_v4b"),"v4b2":score_features(v4b2_candidates,"hybrid_v4b_2")}
features_v3=[x[0] for x in scores["v3"][:500]]
features_v32_500=[x[0] for x in scores["v32"][:500]];features_v32_1000=[x[0] for x in scores["v32"][:1000]]
features_v4b=[x[0] for x in scores["v4b"][:500]]
features_v4b2_500=[x[0] for x in scores["v4b2"][:500]];features_v4b2_1000=[x[0] for x in scores["v4b2"][:1000]]
selection_audit={"v3":len(features_v3),"v32_500":len(features_v32_500),"v32_1000":len(features_v32_1000),"v4b":len(features_v4b),"v4b2_500":len(features_v4b2_500),"v4b2_1000":len(features_v4b2_1000),"v4b2_500_memory":sum(is_memory(c) for c in features_v4b2_500),"v4b2_500_opmodel":sum(is_opmodel(c) for c in features_v4b2_500),"v4b2_1000_memory":sum(is_memory(c) for c in features_v4b2_1000),"v4b2_1000_opmodel":sum(is_opmodel(c) for c in features_v4b2_1000)}
(OUTPUT_ROOT/"feature_selection_audit_ver04.json").write_text(json.dumps(selection_audit,indent=2),encoding="utf-8");print(selection_audit)


## Phase 6: Matched V3, V3.2, V4B, and V4B.2 grouped evaluation


In [ ]:
# =============================================================================
# PHASE 6: matched V3 / V3.2 / V4B / V4B.2 evaluation
# =============================================================================
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score,roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_sample_weight
groups=(enriched.aircraft_identifier.astype("string").fillna(enriched.engine_serial.astype(str)) if "aircraft_identifier" in enriched.columns and enriched.aircraft_identifier.notna().sum()>0 else enriched.engine_serial.astype(str));y=enriched[TARGET].astype(int)
positive_groups=int(pd.DataFrame({"g":groups,"y":y}).groupby("g").y.max().sum());splits=list(GroupKFold(n_splits=min(5,positive_groups)).split(enriched,y,groups))
def evaluate(features,tag):
    X=enriched[features].astype("float32");oof=np.full(len(enriched),np.nan);rows=[]
    for fold,(ti,vi) in enumerate(splits,1):
        Xt,Xv=X.iloc[ti],X.iloc[vi];yt,yv=y.iloc[ti],y.iloc[vi];usable=[c for c in features if Xt[c].notna().sum()>0 and Xt[c].nunique(dropna=True)>1]
        model=HistGradientBoostingClassifier(learning_rate=.07,max_iter=100,max_leaf_nodes=31,min_samples_leaf=50,l2_regularization=1.,early_stopping=True,validation_fraction=.1,n_iter_no_change=10,random_state=RANDOM_STATE+fold)
        model.fit(Xt[usable],yt,sample_weight=compute_sample_weight(class_weight="balanced",y=yt));p=model.predict_proba(Xv[usable])[:,1];oof[vi]=p
        rows.append({"model":tag,"fold":fold,"features_used":len(usable),"roc_auc":roc_auc_score(yv,p),"pr_auc":average_precision_score(yv,p),"prevalence":yv.mean()})
    valid=np.isfinite(oof);s={"model":tag,"rows":int(valid.sum()),"positive_rows":int(y[valid].sum()),"features_used":len(features),"memory_selected":sum(is_memory(c) for c in features),"opmodel_selected":sum(is_opmodel(c) for c in features),"roc_auc":roc_auc_score(y[valid],oof[valid]),"pr_auc":average_precision_score(y[valid],oof[valid]),"prevalence":y[valid].mean()};s["pr_lift"]=s["pr_auc"]/s["prevalence"];return s,pd.DataFrame(rows)
summary_v3,fv3=evaluate(features_v3,"HYBRID_V3")
summary_v32_500,fv325=evaluate(features_v32_500,"HYBRID_V3_2_TOP500")
summary_v32_1000,fv321=evaluate(features_v32_1000,"HYBRID_V3_2_TOP1000")
summary_v4b,fv4b=evaluate(features_v4b,"HYBRID_V4B")
summary_v4b2_500,fvb25=evaluate(features_v4b2_500,"HYBRID_V4B_2_TOP500")
summary_v4b2_1000,fvb21=evaluate(features_v4b2_1000,"HYBRID_V4B_2_TOP1000")
folds=pd.concat([fv3,fv325,fv321,fv4b,fvb25,fvb21],ignore_index=True);folds["pr_lift"]=folds.pr_auc/folds.prevalence;folds.to_csv(OUTPUT_ROOT/"fold_results_v4b2_ver04.csv",index=False)
parents=[summary_v32_500,summary_v32_1000,summary_v4b];children=[summary_v4b2_500,summary_v4b2_1000];best_parent=max(parents,key=lambda z:z["pr_auc"]);best_child=max(children,key=lambda z:z["pr_auc"])
delta={"roc_auc":best_child["roc_auc"]-best_parent["roc_auc"],"pr_auc":best_child["pr_auc"]-best_parent["pr_auc"],"relative_pr_lift_pct":100*(best_child["pr_lift"]/best_parent["pr_lift"]-1)}
comparison={"hybrid_v3":summary_v3,"hybrid_v3_2_top500":summary_v32_500,"hybrid_v3_2_top1000":summary_v32_1000,"hybrid_v4b":summary_v4b,"hybrid_v4b2_top500":summary_v4b2_500,"hybrid_v4b2_top1000":summary_v4b2_1000,"best_parent_by_pr_auc":best_parent["model"],"best_child_by_pr_auc":best_child["model"],"delta_best_v4b2_minus_best_parent":delta,"promotion_gate":{"roc_gain_at_least_0_01":delta["roc_auc"]>=.01,"relative_pr_lift_gain_at_least_5pct":delta["relative_pr_lift_pct"]>=5},"all_code_audit":all_code_audit,"operating_model_audit":operating_model_audit,"method_note":"Exploratory global feature screening; repeat fold-local or out-of-time before promotion."};comparison["promotion_gate"]["exploratory_gate_pass"]=any(comparison["promotion_gate"].values())
(OUTPUT_ROOT/"run_summary_v4b2_ver04.json").write_text(json.dumps(comparison,indent=2),encoding="utf-8");print(json.dumps(comparison,indent=2));print("PASS: Hybrid V4B.2 benchmark completed")
